## Python script to perform audio watermark embedding/detection using the direct-sequence spread spectrum (DSSS) method.


### Preliminary steps
- import dependencies
- Create `files` directory
- preprocess `.mp3` file

In [1]:
#  Required packages
import os
import numpy as np
from scipy.io import wavfile
from pydub import AudioSegment

# Create files directory
output_directory = "./files"
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
    print(f"Directory '{output_directory}' created.")

# Preprocess the .mp3 file
audio = AudioSegment.from_mp3("raw.mp3")
audio = audio.set_channels(1)
audio.export("./files/preprocessed.wav", format="wav")

<_io.BufferedRandom name='./files/preprocessed.wav'>

### Watermaking single audio file using plaintext watermark

In [2]:
import os
import numpy as np
from scipy.io import wavfile

# ==============================================================================
# 1. CONFIGURATION & FILE PATHS
# ==============================================================================
FILES_DIR = "./files"
HOST_SIGNAL_FILE = os.path.join(FILES_DIR, "preprocessed.wav")
WATERMARK_SIGNAL_FILE = os.path.join(FILES_DIR, "wmarked_file.wav")
PSEUDO_RAND_FILE = os.path.join(FILES_DIR, "pseudo_rand.dat")
WATERMARK_ORIGINAL_FILE = os.path.join(FILES_DIR, "watermark_binary.dat")
WATERMARK_EXTENDED_FILE = os.path.join(FILES_DIR, "watermark_extended.dat")

WATERMARK_TEXT = "Goweki"

# Watermarking Parameters
REP_CODE = True          # Enable repetition coding for robustness
# FRAME_LENGTH = 1024      # Frame size (samples)
# CONTROL_STRENGTH = 0.03  # Embedding intensity (alpha)
OVERLAP = 0.0            # Overlap between frames (0.0 = no overlap)
# NUM_REPS = 3             # Repetitions per watermark bit if REP_CODE is True

# # Updated settings for better blind detection performance
FRAME_LENGTH = 4096     # Increased frame size (gives stronger correlation peak)
CONTROL_STRENGTH = 0.15 # Increased embedding strength
NUM_REPS = 5            # More repetitions to filter out host signal noise


# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def fix(xs: float) -> float:
    """Emulates MATLAB's 'fix' function (rounds towards zero)."""
    return np.floor(xs) if xs >= 0 else np.ceil(xs)


def ensure_dirs():
    """Ensure the target directory exists before writing files."""
    os.makedirs(FILES_DIR, exist_ok=True)


# ==============================================================================
# 3. EMBEDDING FUNCTION
# ==============================================================================
def embed():
    """Embeds the text watermark into the host audio file."""
    ensure_dirs()

    # 1. Generate 1D Pseudo-Random Sequence (PRS) in range [-0.5, 0.5]
    prs = np.random.rand(FRAME_LENGTH) - 0.5
    with open(PSEUDO_RAND_FILE, "w") as f:
        f.writelines(f"{d:.6f}\n" for d in prs)

    # 2. Read Host Audio Signal
    sr, host_signal = wavfile.read(HOST_SIGNAL_FILE)

    # Convert stereo to mono & force float64 processing
    if host_signal.ndim > 1:
        host_signal = host_signal[:, 0]
    host_signal = host_signal.astype(np.float64)

    signal_len = len(host_signal)
    frame_shift = int(FRAME_LENGTH * (1 - OVERLAP))
    overlap_length = int(FRAME_LENGTH * OVERLAP)

    # Calculate embedding capacity
    embed_nbit = fix((signal_len - overlap_length) / frame_shift)

    if REP_CODE:
        effective_nbit = np.floor(embed_nbit / NUM_REPS)
        embed_nbit = effective_nbit * NUM_REPS
    else:
        effective_nbit = embed_nbit

    effective_nbit = int(effective_nbit)
    embed_nbit = int(embed_nbit)

    # 3. Convert Watermark Text to Binary & Align/Pad Right
    binary_str = "".join(format(ord(char), "08b") for char in WATERMARK_TEXT)
    if len(binary_str) > effective_nbit:
        binary_str = binary_str[:effective_nbit]
    else:
        binary_str = binary_str.ljust(effective_nbit, "0")

    wmark_original = np.array([int(digit) for digit in binary_str])
    with open(WATERMARK_ORIGINAL_FILE, "w") as f:
        f.writelines(f"{d}\n" for d in wmark_original)

    # Apply Repetition Code if enabled
    wmark_extended = np.repeat(wmark_original, NUM_REPS) if REP_CODE else wmark_original
    with open(WATERMARK_EXTENDED_FILE, "w") as f:
        f.writelines(f"{d}\n" for d in wmark_extended)

    # 4. Embed Frame-by-Frame
    pointer = 0
    wmed_signal = np.zeros(frame_shift * embed_nbit)

    for i in range(embed_nbit):
        frame = host_signal[pointer : pointer + FRAME_LENGTH].copy()
        alpha = CONTROL_STRENGTH * np.max(np.abs(frame))

        # Modulate frame with PRS based on watermark bit
        if wmark_extended[i] == 1:
            frame += alpha * prs
        else:
            frame -= alpha * prs

        wmed_signal[frame_shift * i : frame_shift * (i + 1)] = frame[:frame_shift]
        pointer += frame_shift

    # Append remaining unmodified tail of the audio
    if len(wmed_signal) < signal_len:
        wmed_signal = np.concatenate((wmed_signal, host_signal[len(wmed_signal) : signal_len]))

    # Clip & Save to 16-bit WAV
    wmed_signal = np.clip(wmed_signal, -32768, 32767).astype(np.int16)
    wavfile.write(WATERMARK_SIGNAL_FILE, sr, wmed_signal)
    print("✓ Watermark embedding complete.")


# ==============================================================================
# 4. DETECTION FUNCTION
# ==============================================================================
def detect():
    """Extracts and evaluates the embedded watermark from the audio."""
    _, host_signal = wavfile.read(HOST_SIGNAL_FILE)
    _, eval_signal = wavfile.read(WATERMARK_SIGNAL_FILE)

    # Convert to mono float arrays
    if host_signal.ndim > 1:
        host_signal = host_signal[:, 0]
    if eval_signal.ndim > 1:
        eval_signal = eval_signal[:, 0]

    host_signal = host_signal.astype(np.float64)
    eval_signal = eval_signal.astype(np.float64)

    signal_len = len(eval_signal)
    frame_shift = int(FRAME_LENGTH * (1 - OVERLAP))
    embed_nbit = int(fix((signal_len - int(FRAME_LENGTH * OVERLAP)) / frame_shift))

    if REP_CODE:
        effective_nbit = int(np.floor(embed_nbit / NUM_REPS))
        embed_nbit = effective_nbit * NUM_REPS
    else:
        effective_nbit = embed_nbit

    # Load references from disk
    with open(WATERMARK_ORIGINAL_FILE, "r") as f:
        wmark_original = np.array([int(w.strip()) for w in f.readlines()])

    with open(PSEUDO_RAND_FILE, "r") as f:
        prs = np.array([float(x.strip()) for x in f.readlines()])

    # Detect Bits using Correlation Peak
    pointer = 0
    detected_bit = np.zeros(embed_nbit)

    for i in range(embed_nbit):
        frame_diff = eval_signal[pointer : pointer + FRAME_LENGTH] - host_signal[pointer : pointer + FRAME_LENGTH]
        comp = np.correlate(frame_diff, prs, "full")
        maxp = np.argmax(np.abs(comp))

        detected_bit[i] = 1 if comp[maxp] >= 0 else 0
        pointer += frame_shift

    # Reconstruct watermark via majority voting/averaging
    if REP_CODE:
        wmark_recovered = np.zeros(effective_nbit)
        for i in range(effective_nbit):
            chunk = detected_bit[i * NUM_REPS : (i + 1) * NUM_REPS]
            wmark_recovered[i] = 1 if (np.sum(chunk) / NUM_REPS) >= 0.5 else 0
    else:
        wmark_recovered = detected_bit

    # Decode Binary to ASCII Text
    recovered_binary = "".join(str(int(b)) for b in wmark_recovered)
    chars = [recovered_binary[i : i + 8] for i in range(0, len(recovered_binary), 8)]

    original_text = "".join(
        chr(int(char, 2)) for char in chars if len(char) == 8 and char != "00000000"
    )

    # Metrics Calculation
    BER = np.sum(np.abs(wmark_recovered - wmark_original)) / effective_nbit * 100
    noise = host_signal[: len(eval_signal)] - eval_signal
    SNR = 10 * np.log10(np.sum(host_signal[: len(eval_signal)] ** 2) / np.sum(noise**2))

    # Output Results
    print("\n--- WATERMARK RESULTS ---")
    print(f"ORIGINAL TEXT  : {WATERMARK_TEXT}")
    print(f"RECOVERED TEXT : {original_text}")
    print(f"Bit Error Rate : {BER:.2f}%")
    print(f"Signal-to-Noise: {SNR:.2f} dB")


# ==============================================================================
# 5. RUN EMBED & DETECT
# ==============================================================================
embed()
detect()

✓ Watermark embedding complete.

--- WATERMARK RESULTS ---
ORIGINAL TEXT  : Goweki
RECOVERED TEXT : Goweki
Bit Error Rate : 0.00%
Signal-to-Noise: 16.76 dB


Extract watermark without source file

In [3]:
import os
import numpy as np
from scipy.io import wavfile

def detect_without_original():
    """Detect the watermark directly from the watermarked audio without host signal."""
    # 1. Read Watermarked Audio Signal
    _, eval_signal = wavfile.read(WATERMARK_SIGNAL_FILE)

    # Convert to mono float array
    if eval_signal.ndim > 1:
        eval_signal = eval_signal[:, 0]
    eval_signal = eval_signal.astype(np.float64)

    signal_len = len(eval_signal)
    frame_shift = int(FRAME_LENGTH * (1 - OVERLAP))
    embed_nbit = int(fix((signal_len - int(FRAME_LENGTH * OVERLAP)) / frame_shift))

    if REP_CODE:
        effective_nbit = int(np.floor(embed_nbit / NUM_REPS))
        embed_nbit = effective_nbit * NUM_REPS
    else:
        effective_nbit = embed_nbit

    # 2. Load Reference Files from Disk
    with open(WATERMARK_ORIGINAL_FILE, "r") as f:
        wmark_original = np.array([int(w.strip()) for w in f.readlines()])

    with open(PSEUDO_RAND_FILE, "r") as f:
        prs = np.array([float(x.strip()) for x in f.readlines()])

    # 3. Perform Blind Correlation Detection
    pointer = 0
    detected_bit = np.zeros(embed_nbit)

    for i in range(embed_nbit):
        frame = eval_signal[pointer : pointer + FRAME_LENGTH]
        
        # Correlate watermarked frame directly with PRS
        comp = np.correlate(frame, prs, "full")
        maxp = np.argmax(np.abs(comp))

        # Decide bit value based on correlation peak polarity
        detected_bit[i] = 1 if comp[maxp] >= 0 else 0
        pointer += frame_shift

    # 4. Majority Vote / Average over Repetitions
    if REP_CODE:
        wmark_recovered = np.zeros(effective_nbit)
        for i in range(effective_nbit):
            chunk = detected_bit[i * NUM_REPS : (i + 1) * NUM_REPS]
            wmark_recovered[i] = 1 if (np.sum(chunk) / NUM_REPS) >= 0.5 else 0
    else:
        wmark_recovered = detected_bit

    # 5. Decode Binary String to Text
    recovered_binary = "".join(str(int(b)) for b in wmark_recovered)
    chars = [recovered_binary[i : i + 8] for i in range(0, len(recovered_binary), 8)]

    original_text = "".join(
        chr(int(char, 2)) for char in chars if len(char) == 8 and char != "00000000"
    )

    # 6. Calculate Bit Error Rate (BER)
    BER = np.sum(np.abs(wmark_recovered - wmark_original)) / effective_nbit * 100

    print("\n--- BLIND DETECTION RESULTS ---")
    print(f"ORIGINAL TEXT  : {WATERMARK_TEXT}")
    print(f"RECOVERED TEXT : {original_text}")
    print(f"Bit Error Rate : {BER:.2f}%")


def main():
    detect_without_original()


if __name__ == "__main__":
    main()


--- BLIND DETECTION RESULTS ---
ORIGINAL TEXT  : Goweki
RECOVERED TEXT : Goweki
Bit Error Rate : 0.00%
